# B6 Stage 0 — 準備・監査・互換検証のみ

Stage1 / Validation / Monitorの実行セルはありません。全セル実行でも探索しません。
この保存は探索条件Freezeではありません。Chat確認後に別の正式Freezeを作成してください。

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
B6_CODE_SHA = os.environ.get('B6_CODE_SHA', '')  # GitHubで確認した準備commitの40桁SHA
DATA_ROOT = os.environ.get('B6_DATA_ROOT', '')
BASELINE = os.environ.get('B6_BASELINE', '')
RUN_STAGE0 = False  # 入力設定後、Stage0だけ実行する場合にTrue
SAVE_TO_DRIVE = False
OUT = Path('/content/b6_stage0')
REPO = Path('/content/b6_repo')


In [ ]:
if RUN_STAGE0:
    assert len(B6_CODE_SHA)==40 and all(c in '0123456789abcdef' for c in B6_CODE_SHA)
    assert DATA_ROOT and BASELINE
    if not REPO.exists():
        subprocess.run(['git','clone','https://github.com/TR-KJ/time-entry-portfolio-lab.git',str(REPO)],check=True)
    assert not subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip(), 'dirty checkout: do not overwrite'
    subprocess.run(['git','fetch','origin',B6_CODE_SHA],cwd=REPO,check=True)
    subprocess.run(['git','checkout','--detach',B6_CODE_SHA],cwd=REPO,check=True)
    assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==B6_CODE_SHA
    sys.path.insert(0,str(REPO/'src/research'))
    from b6.stage0 import run
    run(Path(DATA_ROOT),OUT,Path(BASELINE))
else:
    print('NOT RUN: explicit Stage0 switch is OFF')


## 監査表示
Source、入力同一性、価格較正、代表照合だけを表示します。固定Baseline全体の成績表は読みません。

In [ ]:
if RUN_STAGE0:
    import pandas as pd
    display(pd.DataFrame(json.loads((REPO/'results/b6/stage0/source_manifest.json').read_text())))
    display(pd.read_csv(OUT/'input_audit.csv'))
    display(pd.read_csv(OUT/'coverage.csv'))
    display(pd.DataFrame(json.loads((OUT/'price_statistics.json').read_text())))
    display(pd.read_csv(OUT/'sl_grid_proposal.csv'))
    display(pd.read_csv(OUT/'representative_reconciliation.csv'))
    print((OUT/'run_status.json').read_text())


In [ ]:
if RUN_STAGE0:
    from b6.rules import structure_count
    config=json.loads((REPO/'research_inputs/b6/proposal.json').read_text())
    print('提案時間構造:',structure_count(config),'SL設定:',5*structure_count(config))
    print('Stage1以降 NOT RUN。絞込み・年別候補成績・選定理由はまだ存在しません。')
    print((REPO/'docs/b6/parameter_decision_register.md').read_text())
    assert not SAVE_TO_DRIVE, 'Driveへの成果物保存はこのNotebookでは実装しません'


## 後続Notebookの出力仕様（今回は未実行）

Discovery専用：候補ID/5SL別N,PF,AvgR,TotalR,DD/年別同指標/近傍/落選理由/50枠抑制先/段階別試行数。
Validation専用：正式Freeze remote SHAを必須入力とし、2024・2025・合算を別表示。準備Notebookから呼ばない。
Colabで数値条件を発案・変更しない。大きなM1や取引ログはGitHubへ保存しない。